In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField, StringType,
    IntegerType, TimestampType
)
from pyspark.sql.functions import (
    sum as spark_sum, count, window, col
)
from datetime import datetime
import pandas as pd
import os
import shutil
import time

# Membuat SparkSession
spark = SparkSession.builder \
    .appName("Tugas10-KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

print("SparkSession berhasil dibuat.")
print("Versi Spark:", spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/11 17:02:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/11 17:02:34 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


SparkSession berhasil dibuat.
Versi Spark: 3.5.9


In [2]:
# Mendefinisikan skema data pesanan
skema_pesanan = StructType([
    StructField("pesanan_id", StringType(), True),
    StructField("nama_cabang", StringType(), True),
    StructField("nama_menu", StringType(), True),
    StructField("qty", IntegerType(), True),
    StructField("harga_satuan", IntegerType(), True),
    StructField("waktu_pesan", TimestampType(), True),
])

# Membuat folder simulasi streaming
folder_tugas = "tugas10_stream"

shutil.rmtree(folder_tugas, ignore_errors=True)
os.makedirs(folder_tugas, exist_ok=True)

# Membaca data secara streaming
df_stream = spark.readStream \
    .format("csv") \
    .schema(skema_pesanan) \
    .option("header", True) \
    .load(folder_tugas)

print("Streaming source berhasil dibuat.")
print("Apakah DataFrame streaming:", df_stream.isStreaming)

Streaming source berhasil dibuat.
Apakah DataFrame streaming: True


In [3]:
# Menambahkan kolom total_harga
df_pesanan = df_stream.withColumn(
    "total_harga",
    col("qty") * col("harga_satuan")
)

# Mengelompokkan data berdasarkan cabang
agregasi_cabang = df_pesanan.groupBy("nama_cabang").agg(
    spark_sum("total_harga").alias("total_pendapatan"),
    count("pesanan_id").alias("jumlah_pesanan")
)

# Menjalankan streaming query
query1 = agregasi_cabang.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_cabang") \
    .start()

print("Streaming agregasi cabang berhasil dijalankan.")

Streaming agregasi cabang berhasil dijalankan.


In [4]:
# Membaca kembali sumber data streaming
df_stream_waktu = spark.readStream \
    .format("csv") \
    .schema(skema_pesanan) \
    .option("header", True) \
    .load(folder_tugas)

# Menambahkan kolom total_harga
df_pesanan_waktu = df_stream_waktu.withColumn(
    "total_harga",
    col("qty") * col("harga_satuan")
)

# Mengelompokkan berdasarkan window waktu 2 menit
agregasi_waktu = df_pesanan_waktu.groupBy(
    window(col("waktu_pesan"), "2 minutes")
).agg(
    spark_sum("total_harga").alias("total_pendapatan")
)

# Menjalankan streaming query kedua
query2 = agregasi_waktu.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_waktu") \
    .start()

print("Streaming agregasi waktu berhasil dijalankan.")

Streaming agregasi waktu berhasil dijalankan.


In [5]:
waktu_simulasi = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

print("Waktu simulasi:", waktu_simulasi)

Waktu simulasi: 2026-11-11 17:03:33


In [7]:
# Batch pertama
pd.DataFrame({
    "pesanan_id": ["P001", "P002", "P003"],
    "nama_cabang": ["Magelang", "Yogyakarta", "Magelang"],
    "nama_menu": ["Kopi Susu", "Roti", "Latte"],
    "qty": [2, 3, 1],
    "harga_satuan": [18000, 12000, 22000],
    "waktu_pesan": [waktu_simulasi] * 3
}).to_csv(f"{folder_tugas}/batch_1.csv", index=False)

print("Batch pertama berhasil dikirim.")

time.sleep(10)

spark.sql("""
    SELECT * FROM agregasi_cabang
    ORDER BY nama_cabang
""").show()

Batch pertama berhasil dikirim.
+-----------+----------------+--------------+
|nama_cabang|total_pendapatan|jumlah_pesanan|
+-----------+----------------+--------------+
|   Magelang|           58000|             2|
| Yogyakarta|           36000|             1|
+-----------+----------------+--------------+



In [8]:
# Batch kedua
pd.DataFrame({
    "pesanan_id": ["P004", "P005"],
    "nama_cabang": ["Solo", "Magelang"],
    "nama_menu": ["Americano", "Espresso"],
    "qty": [2, 2],
    "harga_satuan": [20000, 25000],
    "waktu_pesan": [waktu_simulasi] * 2
}).to_csv(f"{folder_tugas}/batch_2.csv", index=False)

print("Batch kedua berhasil dikirim.")

time.sleep(10)

spark.sql("""
    SELECT * FROM agregasi_cabang
    ORDER BY nama_cabang
""").show()

Batch kedua berhasil dikirim.
+-----------+----------------+--------------+
|nama_cabang|total_pendapatan|jumlah_pesanan|
+-----------+----------------+--------------+
|   Magelang|          108000|             3|
|       Solo|           40000|             1|
| Yogyakarta|           36000|             1|
+-----------+----------------+--------------+



In [9]:
# Batch ketiga
pd.DataFrame({
    "pesanan_id": ["P006", "P007", "P008"],
    "nama_cabang": ["Yogyakarta", "Solo", "Magelang"],
    "nama_menu": ["Cappuccino", "Latte", "Kopi Susu"],
    "qty": [3, 1, 1],
    "harga_satuan": [22000, 24000, 18000],
    "waktu_pesan": [waktu_simulasi] * 3
}).to_csv(f"{folder_tugas}/batch_3.csv", index=False)

print("Batch ketiga berhasil dikirim.")

time.sleep(10)

spark.sql("""
    SELECT * FROM agregasi_cabang
    ORDER BY nama_cabang
""").show()

Batch ketiga berhasil dikirim.
+-----------+----------------+--------------+
|nama_cabang|total_pendapatan|jumlah_pesanan|
+-----------+----------------+--------------+
|   Magelang|          126000|             4|
|       Solo|           64000|             2|
| Yogyakarta|          102000|             2|
+-----------+----------------+--------------+



In [10]:
spark.sql("""
    SELECT
        window.start,
        window.end,
        total_pendapatan
    FROM agregasi_waktu
""").show(truncate=False)

+-------------------+-------------------+----------------+
|start              |end                |total_pendapatan|
+-------------------+-------------------+----------------+
|2026-11-11 17:02:00|2026-11-11 17:04:00|292000          |
+-------------------+-------------------+----------------+



In [11]:
# Membersihkan folder output dan checkpoint lama
shutil.rmtree("tugas10_output", ignore_errors=True)
shutil.rmtree("tugas10_checkpoint", ignore_errors=True)

# Membuat streaming query ke format Parquet
query3 = df_stream.writeStream \
    .format("parquet") \
    .option("path", "tugas10_output") \
    .option("checkpointLocation", "tugas10_checkpoint") \
    .outputMode("append") \
    .start()

print("Streaming ke file Parquet berhasil dijalankan.")

Streaming ke file Parquet berhasil dijalankan.


In [12]:
# Membuat batch keempat untuk pengujian Parquet
pd.DataFrame({
    "pesanan_id": ["P009"],
    "nama_cabang": ["Solo"],
    "nama_menu": ["Mocha"],
    "qty": [2],
    "harga_satuan": [25000],
    "waktu_pesan": [waktu_simulasi]
}).to_csv(f"{folder_tugas}/batch_4.csv", index=False)

print("Batch keempat berhasil dikirim.")

# Menunggu Spark memproses data
time.sleep(10)

Batch keempat berhasil dikirim.


In [13]:
!ls tugas10_output

part-00000-0d56283b-d300-4c27-969e-d52e5e1aff5a-c000.snappy.parquet
part-00000-65d7c9d1-5cbb-4f04-b7b9-b0b4b1f0b1ed-c000.snappy.parquet
part-00001-dda30941-f754-43d6-93cd-ba4105415c02-c000.snappy.parquet
part-00002-52218139-c932-491d-ae6d-976a2450119e-c000.snappy.parquet
_spark_metadata


In [14]:
# Membaca file Parquet sebagai DataFrame batch
df_hasil_parquet = spark.read.parquet("tugas10_output")

# Menampilkan data yang berhasil dibaca
df_hasil_parquet.show(truncate=False)

# Menampilkan struktur kolom
df_hasil_parquet.printSchema()

+----------+-----------+----------+---+------------+-------------------+
|pesanan_id|nama_cabang|nama_menu |qty|harga_satuan|waktu_pesan        |
+----------+-----------+----------+---+------------+-------------------+
|P006      |Yogyakarta |Cappuccino|3  |22000       |2026-11-11 17:03:33|
|P007      |Solo       |Latte     |1  |24000       |2026-11-11 17:03:33|
|P008      |Magelang   |Kopi Susu |1  |18000       |2026-11-11 17:03:33|
|P001      |Magelang   |Kopi Susu |2  |18000       |2026-11-11 17:03:33|
|P002      |Yogyakarta |Roti      |3  |12000       |2026-11-11 17:03:33|
|P003      |Magelang   |Latte     |1  |22000       |2026-11-11 17:03:33|
|P004      |Solo       |Americano |2  |20000       |2026-11-11 17:03:33|
|P005      |Magelang   |Espresso  |2  |25000       |2026-11-11 17:03:33|
|P009      |Solo       |Mocha     |2  |25000       |2026-11-11 17:03:33|
+----------+-----------+----------+---+------------+-------------------+

root
 |-- pesanan_id: string (nullable = true)
 |-